# Feature Engineering

Feature engineering is the process of creating new variables and transforming existing ones to improve the predictive performance of machine learning models.

This notebook focuses on generating meaningful features from the flight dataset while preventing data leakage. Variables that would reveal the outcome of the flight after departure will be identified and excluded from the final modeling dataset.

The resulting dataset will contain only information that would realistically be available before or at the scheduled departure time.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

In [ ]:
from pathlib import Path


def find_project_root() -> Path:
    """
    Locate the project root whether the notebook is launched
    from the project root or from inside the notebooks folder.
    """
    current_path = Path.cwd().resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "requirements.txt").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root could not be located. "
        "Run this notebook from inside the "
        "Flight_Delay_Prediction project folder."
    )


PROJECT_ROOT = find_project_root()

RAW_DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

RAW_DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
MODELS_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
preprocessed_data_path = (
    PROCESSED_DATA_DIRECTORY
    / "preprocessed_flight_data.csv"
)

df = pd.read_csv(
    preprocessed_data_path,
    parse_dates=["FL_DATE"]
)
df.head()

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

### 📝 Interpretation

The preprocessed dataset was loaded successfully. It contains the cleaned observations from the previous notebook and will serve as the foundation for feature engineering.

In [ ]:
# Year
df["YEAR"] = df["FL_DATE"].dt.year

# Month
df["MONTH"] = df["FL_DATE"].dt.month

# Day
df["DAY"] = df["FL_DATE"].dt.day

# Day of week
df["DAY_OF_WEEK"] = df["FL_DATE"].dt.day_name()

# Quarter
df["QUARTER"] = df["FL_DATE"].dt.quarter

# Weekend
df["IS_WEEKEND"] = df["DAY_OF_WEEK"].isin(
    ["Saturday", "Sunday"]
).astype(int)

In [ ]:
df[
    [
        "FL_DATE",
        "YEAR",
        "MONTH",
        "DAY",
        "DAY_OF_WEEK",
        "QUARTER",
        "IS_WEEKEND"
    ]
].head()

### 📝 Interpretation

Several calendar-based features were extracted from the flight date.

These variables may capture seasonal patterns, weekday effects, holiday travel trends, and operational differences between weekdays and weekends that influence flight delays.

In [ ]:
df["DEP_HOUR"] = (
    df["CRS_DEP_TIME"] // 100
).astype(int)

In [ ]:
df[["CRS_DEP_TIME", "DEP_HOUR"]].head(10)

### 📝 Interpretation

The scheduled departure hour was extracted from the scheduled departure time.

This feature allows the model to learn whether flights departing during certain hours of the day are more likely to experience delays.

# Time of Day

The scheduled departure hour is grouped into broader time-of-day categories.

These categories may capture operational patterns such as morning traffic peaks, afternoon congestion, and overnight operations, which can influence flight delays.

In [ ]:
# Create Time of Day feature

def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df["TIME_OF_DAY"] = df["DEP_HOUR"].apply(get_time_of_day)

In [ ]:
df[["DEP_HOUR", "TIME_OF_DAY"]].head(10)

### 📝 Interpretation

The scheduled departure hour was transformed into a categorical feature representing the time of day.

This transformation reduces the number of unique values while preserving meaningful operational patterns that may influence flight delays.

# Season

Seasonal weather conditions and travel demand vary throughout the year.

A new feature representing the season is created from the flight month to capture these patterns.

In [ ]:
# Create Season feature

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

df["SEASON"] = df["MONTH"].apply(get_season)

In [ ]:
df[["MONTH", "SEASON"]].head(12)

### 📝 Interpretation

The flight month was grouped into four seasons.

Seasonal variation can influence flight operations due to weather conditions, passenger demand, and airport congestion.

# Distance Category

Flights are grouped into distance categories based on the total travel distance.

Categorizing flight distance may help the model identify differences between short-haul, medium-haul, and long-haul operations.

In [ ]:
# Create Distance Category

def distance_category(distance):
    if distance < 500:
        return "Short"
    elif distance < 1500:
        return "Medium"
    else:
        return "Long"

df["DISTANCE_CATEGORY"] = df["DISTANCE"].apply(distance_category)

In [ ]:
df[["DISTANCE", "DISTANCE_CATEGORY"]].head(10)

### 📝 Interpretation

The continuous flight distance was converted into three operational categories.

This transformation allows the model to capture broad differences between short-, medium-, and long-haul flights while reducing sensitivity to small variations in distance.

# Route Feature

A new feature representing the flight route is created by combining the origin and destination airports.

Different routes may experience different weather conditions, traffic volumes, and airport operational characteristics, making this feature potentially valuable for prediction.

In [ ]:
# Create Route

df["ROUTE"] = df["ORIGIN"] + "_" + df["DEST"]

In [ ]:
df[["ORIGIN", "DEST", "ROUTE"]].head()

### 📝 Interpretation

A new route feature was created by combining the origin and destination airport codes.

This feature uniquely identifies each flight route and may help the model learn route-specific delay patterns.

# Data Leakage Analysis

Data leakage occurs when a machine learning model is trained using information that would not be available at the time a prediction is made.

The objective of this project is to predict whether a flight will experience a significant arrival delay **before the flight is completed**. Therefore, any variables generated during or after the flight must be excluded from the modeling dataset.

The following section identifies variables that introduce data leakage and explains why they cannot be used as predictors.

In [ ]:
leakage_features = pd.DataFrame({
    "Feature": [
        "ARR_TIME",
        "ARR_DELAY",
        "WHEELS_ON",
        "TAXI_IN",
        "ELAPSED_TIME",
        "AIR_TIME",
        "DELAY_DUE_CARRIER",
        "DELAY_DUE_WEATHER",
        "DELAY_DUE_NAS",
        "DELAY_DUE_SECURITY",
        "DELAY_DUE_LATE_AIRCRAFT",
        "CANCELLATION_CODE"
    ],
    "Reason for Removal": [
        "Known only after arrival.",
        "Target variable used to create IS_DELAYED.",
        "Recorded after landing.",
        "Recorded after landing.",
        "Known after flight completion.",
        "Known after flight completion.",
        "Calculated after the delay occurs.",
        "Calculated after the delay occurs.",
        "Calculated after the delay occurs.",
        "Calculated after the delay occurs.",
        "Calculated after the delay occurs.",
        "Available only for cancelled flights."
    ]
})

leakage_features

### 📝 Interpretation

Several variables contain information that becomes available only after the flight has departed or arrived.

Including these variables during model training would allow the model to indirectly observe the outcome it is attempting to predict, resulting in overly optimistic performance and poor real-world generalization.

Therefore, these features will be removed before model development.

# Remove Leakage Features

The identified leakage variables are removed from the dataset to ensure that only information available before or at the scheduled departure time is used for prediction.

In [ ]:
# Remove post-arrival, leakage, cancellation, and diversion columns

columns_to_drop = [
    "ARR_TIME",
    "ARR_DELAY",
    "WHEELS_ON",
    "TAXI_IN",
    "ELAPSED_TIME",
    "AIR_TIME",
    "TAXI_OUT",
    "WHEELS_OFF",
    "DELAY_DUE_CARRIER",
    "DELAY_DUE_WEATHER",
    "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY",
    "DELAY_DUE_LATE_AIRCRAFT",
    "CANCELLATION_CODE",
    "CANCELLED",
    "DIVERTED"
]

df_model = df.drop(
    columns=columns_to_drop,
    errors="raise"
)

print("Modeling dataset shape:")
print(df_model.shape)

print("\nRemaining columns:")
print(df_model.columns.tolist())

### 📝 Interpretation

The leakage features were successfully removed from the dataset.

The resulting dataset now contains only variables that are suitable for predictive modeling and can realistically be used to estimate flight delays before the flight is completed.

# Peak Travel Season

Certain months experience significantly higher passenger demand due to holidays and vacation periods.

A binary feature is created to identify flights occurring during peak travel seasons.

In [ ]:
# Peak travel season

df_model["IS_PEAK_SEASON"] = (
    df_model["MONTH"].isin([6, 7, 8, 11, 12])
).astype(int)

df_model["IS_PEAK_SEASON"].value_counts()

# Busy Departure Hours

Airport congestion tends to be higher during the morning and evening travel peaks.

A binary feature is created to identify flights scheduled during these busy operating periods.

In [ ]:
# Busy departure hours

df_model["IS_BUSY_HOUR"] = (
    df_model["DEP_HOUR"].between(7, 10) |
    df_model["DEP_HOUR"].between(16, 19)
).astype(int)

df_model["IS_BUSY_HOUR"].value_counts()

### 📝 Interpretation

Two additional binary features were created using aviation domain knowledge.

`IS_PEAK_SEASON` identifies flights during months with historically higher travel demand, while `IS_BUSY_HOUR` identifies departures scheduled during typical airport congestion periods.

These features may improve the model's ability to capture operational patterns associated with flight delays.

# Save the Engineered Dataset

The dataset is saved after completing feature engineering and removing leakage variables.

This version will be used for exploratory data analysis and model development.

In [ ]:
engineered_output_path = (
    PROCESSED_DATA_DIRECTORY
    / "engineered_flight_data.csv"
)

df_model.to_csv(
    engineered_output_path,
    index=False
)

print("Engineered dataset saved to:")
print(engineered_output_path)

print("\nSaved dataset shape:")
print(df_model.shape)

print("\nSaved engineered columns:")
print(df_model.columns.tolist())

# 📌 Notebook Summary

In this notebook, new predictive features were engineered from the original dataset.

The following tasks were completed:

- Extracted calendar-based features.
- Created departure time categories.
- Created seasonal features.
- Generated route and distance-based features.
- Added aviation domain knowledge features.
- Identified and removed data leakage variables.
- Saved the engineered dataset for subsequent analysis and model development.

The dataset is now ready for Exploratory Data Analysis (EDA), where feature relationships and patterns associated with flight delays will be investigated.